# Movie ABT Enrichment with TMDB and IMDB Data

## Packages Used
**ONLY packages from your Notebook 3 - NO beautifulsoup4, NO requests**

- langchain
- langchain-community (includes WebBaseLoader)
- langgraph
- python-dotenv
- openpyxl
- pandas
- langchain-openai

## What it does:
1. Reads your ABT CSV/Excel file
2. For each movie, fetches data from TMDB using tmdbId
3. For each movie, fetches data from IMDB using imdbId
4. Adds enriched attributes to the ABT
5. Saves enhanced ABT as Excel file

## ID Formatting:
- TMDB: Integer (9091, not 9091.0)
- IMDB: With 'tt' prefix (tt0114576, not 114576)

In [ ]:
# Install packages - SAME AS YOUR NOTEBOOK 3
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai --quiet

print("✓ Packages installed")

In [1]:
# Import libraries - USING YOUR PACKAGES ONLY
import os
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, Optional

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_community.document_loaders import WebBaseLoader

print("✓ Imports successful")

✓ Imports successful


In [2]:
# Load environment variables
load_dotenv()
load_dotenv('../.env')

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✓ LLM initialized")

✓ LLM initialized


## Configuration

In [ ]:
# Configuration
INPUT_FILE = "analytics_base_table_with_genres_list.csv"
OUTPUT_FILE = "analytics_base_table_ENRICHED.xlsx"

# How many movies to process
MAX_MOVIES = None  # Process all
# MAX_MOVIES = 10  # For testing

# Rate limiting (seconds between requests)
SLEEP_BETWEEN_REQUESTS = 1.5

print("✓ Configuration loaded")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Max movies: {MAX_MOVIES if MAX_MOVIES else 'All'}")

## Data Fetching Functions

Using **WebBaseLoader** from your Notebook 3 (NOT beautifulsoup4)

In [ ]:
def fetch_tmdb_data(tmdb_id: str) -> Optional[Dict]:
    """
    Fetch movie data from TMDB using WebBaseLoader.
    
    Returns dict with tmdb_ prefixed fields or None if failed.
    """
    if pd.isna(tmdb_id) or str(tmdb_id).strip() == '':
        return None
    
    try:
        url = f"https://www.themoviedb.org/movie/{tmdb_id}"
        
        # Use WebBaseLoader - from YOUR Notebook 3
        loader = WebBaseLoader([url])
        docs = loader.load()
        
        if not docs or not docs[0].page_content:
            return None
        
        # Use LLM to extract structured data
        extraction_prompt = f"""
Extract movie information from this TMDB page and return as JSON.

Extract these fields (use null if not found):
- title: Movie title
- original_title: Original title if different
- overview: Plot synopsis/description
- tagline: Movie tagline
- status: Release status (Released, Planned, etc.)
- release_date: Release date (YYYY-MM-DD format)
- runtime: Runtime in minutes (number only)
- budget: Budget in dollars (number only)
- revenue: Revenue in dollars (number only)
- popularity: Popularity score (number)
- genres: List of genre names
- original_language: Original language code (e.g., 'en')
- spoken_languages: List of spoken language names

Page Content:
{docs[0].page_content[:8000]}

Return ONLY valid JSON, no other text.
"""
        
        result = llm.invoke([HumanMessage(content=extraction_prompt)])
        
        # Parse JSON from response
        json_str = result.content.strip()
        if json_str.startswith('```json'):
            json_str = json_str.split('```json')[1].split('```')[0].strip()
        elif json_str.startswith('```'):
            json_str = json_str.split('```')[1].split('```')[0].strip()
        
        data = json.loads(json_str)
        
        # Prefix all fields with tmdb_
        return {f"tmdb_{k}": v for k, v in data.items()}
        
    except Exception as e:
        print(f"  ⚠ TMDB fetch failed for {tmdb_id}: {e}")
        return None


def fetch_imdb_data(imdb_id: str) -> Optional[Dict]:
    """
    Fetch movie data from IMDB using WebBaseLoader.
    
    Returns dict with imdb_ prefixed fields or None if failed.
    """
    if pd.isna(imdb_id) or str(imdb_id).strip() == '':
        return None
    
    try:
        url = f"https://www.imdb.com/title/{imdb_id}/"
        
        # Use WebBaseLoader - from YOUR Notebook 3
        loader = WebBaseLoader([url])
        docs = loader.load()
        
        if not docs or not docs[0].page_content:
            return None
        
        # Use LLM to extract structured data
        extraction_prompt = f"""
Extract movie information from this IMDB page and return as JSON.

Extract these fields (use null if not found):
- id: IMDB ID (e.g., 'tt0113041')
- title: Movie title
- genres: List of genre names
- rating: IMDB rating (number, e.g., 7.5)
- cast: List of main cast member names (top 5-10)
- overview: Plot synopsis
- runtime: Runtime in minutes (number only)
- release_year: Release year (number, e.g., 1995)
- poster_url: URL to movie poster image
- keywords: List of keywords/tags

Page Content:
{docs[0].page_content[:8000]}

Return ONLY valid JSON, no other text.
"""
        
        result = llm.invoke([HumanMessage(content=extraction_prompt)])
        
        # Parse JSON from response
        json_str = result.content.strip()
        if json_str.startswith('```json'):
            json_str = json_str.split('```json')[1].split('```')[0].strip()
        elif json_str.startswith('```'):
            json_str = json_str.split('```')[1].split('```')[0].strip()
        
        data = json.loads(json_str)
        
        # Prefix all fields with imdb_
        return {f"imdb_{k}": v for k, v in data.items()}
        
    except Exception as e:
        print(f"  ⚠ IMDB fetch failed for {imdb_id}: {e}")
        return None


print("✓ Fetch functions defined (using WebBaseLoader from your Notebook 3)")

## Load ABT Data

In [ ]:
# Read input file
input_path = Path(INPUT_FILE)

if not input_path.exists():
    input_path = Path("../data") / INPUT_FILE
    if not input_path.exists():
        raise FileNotFoundError(f"Could not find {INPUT_FILE}")

print(f"Reading: {input_path}")

# Read based on extension
if input_path.suffix == '.csv':
    df = pd.read_csv(input_path)
else:
    df = pd.read_excel(input_path)

print(f"✓ Loaded {len(df)} movies")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(3)

In [ ]:
# Limit to MAX_MOVIES if specified
if MAX_MOVIES:
    df = df.head(MAX_MOVIES)
    print(f"✓ Limited to {len(df)} movies for processing")

## Initialize New Columns

In [ ]:
# TMDB columns
tmdb_columns = [
    'tmdb_title', 'tmdb_original_title', 'tmdb_overview', 'tmdb_tagline',
    'tmdb_status', 'tmdb_release_date', 'tmdb_runtime', 'tmdb_budget',
    'tmdb_revenue', 'tmdb_popularity', 'tmdb_genres',
    'tmdb_original_language', 'tmdb_spoken_languages'
]

# IMDB columns
imdb_columns = [
    'imdb_id', 'imdb_title', 'imdb_genres', 'imdb_rating',
    'imdb_cast', 'imdb_overview', 'imdb_runtime',
    'imdb_release_year', 'imdb_poster_url', 'imdb_keywords'
]

# Add columns to dataframe
for col in tmdb_columns + imdb_columns:
    if col not in df.columns:
        df[col] = None

print(f"✓ Added {len(tmdb_columns + imdb_columns)} new columns")

## Enrich Data from TMDB and IMDB

**With corrected ID formatting:**
- TMDB ID: Integer (9091, not 9091.0)
- IMDB ID: With 'tt' prefix (tt0114576, not 114576)

In [ ]:
# Process each movie with CORRECTED ID formatting
print("="*70)
print("ENRICHING MOVIES WITH TMDB AND IMDB DATA")
print("="*70)
print(f"Processing {len(df)} movies...\n")

start_time = time.time()
success_count = {'tmdb': 0, 'imdb': 0}
fail_count = {'tmdb': 0, 'imdb': 0}

for idx, row in df.iterrows():
    movie_title = row.get('title', 'Unknown')
    
    # ═══════════════════════════════════════════════════════
    # CORRECTED ID FORMATTING
    # ═══════════════════════════════════════════════════════
    
    # TMDB ID - integer without .0
    tmdb_id = row.get('tmdbId')
    if pd.notna(tmdb_id):
        tmdb_id_display = int(tmdb_id)  # For display: 9091
        tmdb_id_api = str(int(tmdb_id))  # For API: "9091"
    else:
        tmdb_id_display = "N/A"
        tmdb_id_api = None
    
    # IMDB ID - with 'tt' prefix
    imdb_id = row.get('imdbId')
    if pd.notna(imdb_id):
        imdb_id_num = str(int(imdb_id)).zfill(7)
        imdb_id_display = f"tt{imdb_id_num}"  # For display: tt0114576
        imdb_id_api = f"tt{imdb_id_num}"  # For API: tt0114576
    else:
        imdb_id_display = "N/A"
        imdb_id_api = None
    
    print(f"\n[{idx+1}/{len(df)}] {movie_title}")
    print(f"TMDB ID: {tmdb_id_display} | IMDB ID: {imdb_id_display}")
    
    # Fetch TMDB data
    if tmdb_id_api:
        print(f"  Fetching TMDB...")
        tmdb_data = fetch_tmdb_data(tmdb_id_api)
        
        if tmdb_data:
            for key, value in tmdb_data.items():
                df.at[idx, key] = value
            print(f"  ✓ TMDB data retrieved")
            success_count['tmdb'] += 1
        else:
            fail_count['tmdb'] += 1
        
        time.sleep(SLEEP_BETWEEN_REQUESTS)
    
    # Fetch IMDB data
    if imdb_id_api:
        print(f"  Fetching IMDB...")
        imdb_data = fetch_imdb_data(imdb_id_api)
        
        if imdb_data:
            for key, value in imdb_data.items():
                df.at[idx, key] = value
            print(f"  ✓ IMDB data retrieved")
            success_count['imdb'] += 1
        else:
            fail_count['imdb'] += 1
        
        time.sleep(SLEEP_BETWEEN_REQUESTS)
    
    # Progress update every 10 movies
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df) - idx - 1) / rate if rate > 0 else 0
        print(f"\n  Progress: {idx+1}/{len(df)} | Rate: {rate:.1f} movies/sec | ETA: {remaining/60:.1f} min")

elapsed_time = time.time() - start_time

print("\n" + "="*70)
print("ENRICHMENT COMPLETE")
print("="*70)
print(f"Total time: {elapsed_time/60:.1f} minutes")
print(f"\nTMDB: {success_count['tmdb']} successful, {fail_count['tmdb']} failed")
print(f"IMDB: {success_count['imdb']} successful, {fail_count['imdb']} failed")

## Preview Enriched Data

In [ ]:
# Show sample
print("Sample of enriched data:")
print("="*70)

sample_movie = df.iloc[0]
print(f"\nMovie: {sample_movie['title']}")
print("\nTMDB Fields:")
for col in tmdb_columns[:5]:
    if col in df.columns:
        print(f"  {col}: {sample_movie[col]}")

print("\nIMDB Fields:")
for col in imdb_columns[:5]:
    if col in df.columns:
        print(f"  {col}: {sample_movie[col]}")

In [ ]:
# Data completeness
print("\nData Completeness:")
print("="*70)

all_new_cols = tmdb_columns + imdb_columns
for col in all_new_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        pct = (non_null / len(df)) * 100
        print(f"{col:30} {non_null:4}/{len(df):4} ({pct:5.1f}%)")

## Save Enriched ABT

In [ ]:
# Determine output path
output_path = input_path.parent / OUTPUT_FILE

# Define all_new_cols for statistics
all_new_cols = tmdb_columns + imdb_columns

print(f"Saving enriched ABT to: {output_path}")

# Save as Excel
df.to_excel(output_path, index=False, engine='openpyxl')

print(f"✓ Saved {len(df)} enriched movies")
print(f"✓ Total columns: {len(df.columns)}")
print(f"  - Original columns: {len(df.columns) - len(all_new_cols)}")
print(f"  - New TMDB columns: {len(tmdb_columns)}")
print(f"  - New IMDB columns: {len(imdb_columns)}")
print(f"\n✓ File saved: {output_path}")

In [ ]:
# Also save CSV backup
csv_output = output_path.with_suffix('.csv')
df.to_csv(csv_output, index=False)
print(f"✓ CSV backup saved: {csv_output}")

## Summary Report

In [ ]:
print("="*70)
print("ENRICHMENT SUMMARY REPORT")
print("="*70)
print(f"\nInput file:  {input_path}")
print(f"Output file: {output_path}")
print(f"\nMovies processed: {len(df)}")
print(f"Processing time:  {elapsed_time/60:.1f} minutes")
print(f"\nSuccess rates:")
print(f"  TMDB: {success_count['tmdb']}/{len(df)} ({success_count['tmdb']/len(df)*100:.1f}%)")
print(f"  IMDB: {success_count['imdb']}/{len(df)} ({success_count['imdb']/len(df)*100:.1f}%)")
print(f"\nNew columns added: {len(all_new_cols)}")
print(f"  TMDB columns: {len(tmdb_columns)}")
print(f"  IMDB columns: {len(imdb_columns)}")
print(f"\nTotal columns in enriched file: {len(df.columns)}")
print("\nPackages used: ONLY from your Notebook 3")
print("  ✓ WebBaseLoader (NO beautifulsoup4)")
print("  ✓ LangChain tools (NO requests)")
print("="*70)

## Done! 🎉

Your enriched ABT file has been saved with comprehensive movie data from TMDB and IMDB.

### Packages Used:
**ONLY packages from your Notebook 3:**
- langchain
- langchain-community (WebBaseLoader)
- langgraph
- python-dotenv
- openpyxl
- pandas
- langchain-openai

**NO unauthorized packages:**
- ❌ NO beautifulsoup4
- ❌ NO requests

### Files Created:
- Excel: Full enriched data
- CSV: Backup copy